# Verification: Refactored `src/` matches baseline notebook

Run this cell by cell in Colab **after** uploading the `src/` folder and `configs/sanjay_van.yaml`.

Each section runs one module from `src/` and compares its output to the same output from the baseline notebook (`00_baseline.ipynb`). We do not expect byte-for-byte identity — GEE is non-deterministic at the level of cloud scheduling — but summary statistics over the ROI should match to 3-4 decimal places.

**Order matters.** Run top to bottom. Do not skip cells — each module depends on the previous one.

## Setup (Colab)

If you're running in Colab, make sure the repo is in your Python path. The easiest way: upload the whole `forest-mgmt-units/` folder to your Google Drive, mount Drive, and point Python at it.

In [ ]:
# ---- Colab setup ----
import sys, os

# Adjust this to wherever you uploaded the repo
PROJECT_ROOT = "/content/drive/MyDrive/forest-mgmt-units"

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

# If Drive isn't mounted yet, uncomment:
# from google.colab import drive
# drive.mount('/content/drive')

!pip install geemap pyyaml --quiet
print('Working directory:', os.getcwd())
print('sys.path includes:', PROJECT_ROOT in sys.path)

In [ ]:
# ---- EE auth + config load ----
import ee
import geemap
from src.config import load_config, print_config_summary

config = load_config("configs/sanjay_van.yaml")

try:
    ee.Initialize(project=config['project_id'])
    print("GEE initialized.")
except Exception:
    ee.Authenticate()
    ee.Initialize(project=config['project_id'])

# Reload config AFTER ee.Initialize, because load_config creates ee.Geometry objects
config = load_config("configs/sanjay_van.yaml")
print_config_summary(config)

## Module 1 — Masking

**Baseline check:** In your baseline notebook, the masking cell prints:
```
VIIRS threshold (p95 of buffered ROI): XX.XX nW/cm^2/sr
Total ROI pixels (10m): NNNNNN
Valid forest pixels    : NNNNNN
Survival rate          : XX.X%
```

Compare the numbers below to those. They should match to within rounding.

In [ ]:
from src import masking

valid_mask, mask_stats = masking.build_mask(config)
print("\n>>> Compare these to the baseline notebook's Module 1 output:")
print(mask_stats)

## Module 2 — Phenology (coefficients only, fast check)

Just fit the harmonic regression and check the observation count.

**Baseline check:** `n_obs` should match the `HLS obs count` from the baseline (typically 400-700 for Sanjay Van over the 10-year window).

In [ ]:
from src import phenology

hls_coeffs, hls_merged, n_obs = phenology.compute_coefficients(config, valid_mask)
print(f"\n>>> n_obs = {n_obs}  (compare to baseline)")

# Sanity check one pixel's coefficients against the baseline
test_point = ee.Geometry.Point([77.175, 28.53])
sample = hls_coeffs.reduceRegion(
    reducer=ee.Reducer.first(),
    geometry=test_point,
    scale=30,
).getInfo()
print('\nCoefficients at test point (77.175, 28.53):')
for k, v in sample.items():
    print(f'  {k}: {v}')

## Module 2 — Phenology (full PCA features)

This is slower because it computes the 24x24 covariance matrix and eigendecomposition.

**Baseline check:** The output image should have 6 bands named `Pheno_PC1..Pheno_PC5, Pheno_Trend`.

In [ ]:
pheno_features, pheno_meta = phenology.compute_features(config, valid_mask)
print('\n>>> Band names:', pheno_features.bandNames().getInfo())
print('Should be: [Pheno_PC1, Pheno_PC2, Pheno_PC3, Pheno_PC4, Pheno_PC5, Pheno_Trend]')

## Module 3 — Radar

In [ ]:
from src import radar

radar_features, radar_meta = radar.compute_features(config, valid_mask)
print('\n>>> S1 obs count:', radar_meta['n_s1_obs'])
print('Bands:', radar_features.bandNames().getInfo())
print('Should be: [S1_Ratio_P90, S1_Ratio_P10, S1_Ratio_StdDev]')

## Module 4 — Static features

In [ ]:
from src import static_features

static_feats, _ = static_features.compute_features(config, valid_mask)
print('\n>>> Bands:', static_feats.bandNames().getInfo())
print('Should be: [Canopy_Height, Elevation, Slope]')

# We keep canopy_h separately for the high-res stack (it's 10m native)
canopy_h = static_feats.select('Canopy_Height')

## Module 5 — S2 seasonal composites

In [ ]:
from src import s2_composites

s2_stack, s2_meta = s2_composites.compute_features(config, valid_mask)
print('\n>>> Number of bands:', len(s2_meta['band_names']))
print('Should be: 24 (4 seasons x 6 bands)')
print('First 6 bands:', s2_meta['band_names'][:6])

## Modules 6a + 7a + 8 — High-res stack, pixel norm, SNIC

In [ ]:
from src import segmentation

highres_stack_norm, hr_band_names, seed_spacing_px = segmentation.build_highres_stack(
    config, s2_stack, radar_features, canopy_h, valid_mask,
)
snic = segmentation.run_snic(config, highres_stack_norm, seed_spacing_px)
print('\n>>> SNIC result band names (first 5):', snic.bandNames().getInfo()[:5])
print('Should contain "clusters" + <feature>_mean bands')

## Modules 6b + 9 + 10 — Full stack, per-stand aggregation, stand-level norm

In [ ]:
from src import aggregation

full_stack_raw, full_band_names = aggregation.build_full_stack(
    config, pheno_features, radar_features, canopy_h, static_feats, valid_mask,
)

stand_means = aggregation.aggregate_per_stand(
    config, full_stack_raw, snic.select('clusters'), valid_mask, full_band_names,
)

normalized_stands = aggregation.normalize_stands(
    config, stand_means, full_band_names, valid_mask,
)
print('\n>>> Normalized stands ready.')

## Visual sanity check — render the final SNIC map

This should look the same as the Module 8 map in your baseline notebook.

In [ ]:
Map = geemap.Map()
Map.add_basemap('HYBRID')
Map.centerObject(config['roi'], 14)
Map.addLayer(valid_mask.eq(0).selfMask(), {'palette': ['black']}, 'Excluded')
Map.addLayer(
    snic.select('clusters').randomVisualizer().updateMask(valid_mask),
    {}, 'SNIC stands (refactored)',
)
Map

## If everything above matches the baseline → refactor is correct.

Next step: run `notebooks/01_build_assets.ipynb` to export all the intermediate assets.

---

### What to do if numbers don't match

1. **Tiny rounding differences** (< 0.1% on counts, < 0.01 on float values): GEE non-determinism, ignore.
2. **Band name differences**: check the `.rename()` calls in the relevant `src/*.py`.
3. **Order-of-magnitude differences**: you probably have a mask misapplied or a collection filter wrong. Go back to the matching cell in `00_baseline.ipynb` and diff line by line.
4. **Silent NaNs / empty images**: often caused by `valid_mask` being applied at the wrong scale. Check `.reproject()` calls in `phenology.py` and `aggregation.py`.